# 01 — Gate de treinamento: overfit em quatro imagens

Este notebook testa o pipeline completo exclusivamente em quatro amostras de treino. Não é resultado científico, não usa validação, teste ou FUSeg.

In [ ]:
from pathlib import Path
import os

from google.colab import drive

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas")
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Projeto: {PROJECT_ROOT}")

## Dependências e saída contínua

In [ ]:
import subprocess
import sys


def run_stream(command, allowed_returncodes=(0,)):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    print(f"\nreturncode={returncode}")
    if returncode not in allowed_returncodes:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


run_stream([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"])

## Ambiente e imports

In [ ]:
run_stream([sys.executable, "scripts/check_environment.py"])

## Cache temporário dos dados

In [ ]:
import shutil

DATA_ROOT = Path("/content/tcc_data")
shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print(DATA_ROOT / "Dataset")

## Testes dos componentes

Loss, gradiente, métricas por imagem, checkpoint, U-Net e proteção da configuração são testados antes do uso da GPU.

In [ ]:
run_stream(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_training_components.py",
        "tests/test_overfit_config.py",
    ]
)

## Executar o overfit

O processo imprime uma linha por época e pode encerrar antes de 200 épocas se Dice ≥ 0,97 for mantido por cinco épocas.

In [ ]:
run_stream(
    [
        sys.executable,
        "scripts/run_overfit_smoke.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--data-root",
        str(DATA_ROOT),
        "--config",
        "configs/overfit_smoke.yaml",
    ]
)

## Mostrar o resultado mais recente

In [ ]:
import json
from IPython.display import Image as DisplayImage, display

runs = sorted((PROJECT_ROOT / "results/smoke/overfit").glob("overfit_smoke_*"))
if not runs:
    raise FileNotFoundError("Nenhuma execução de overfit encontrada")
LATEST_RUN = runs[-1]
summary = json.loads((LATEST_RUN / "final_summary.json").read_text())
display(summary)
display(DisplayImage(filename=str(LATEST_RUN / "overfit_panel.png")))
print(f"Resultados: {LATEST_RUN}")

## Estabilização formal após o primeiro resultado

Use esta célula quando o primeiro treino tiver alcançado Dice ≥ 0,97 apenas pontualmente. Ela carrega o melhor checkpoint, reduz o learning rate para 0,0001 e exige cinco épocas consecutivas acima da meta. Uma nova pasta é criada; a execução original não é sobrescrita. O código de saída 2 apenas indica que o gate ainda não foi atingido e não apaga os artefatos.

In [ ]:
source_runs = [
    path
    for path in (PROJECT_ROOT / "results/smoke/overfit").glob("overfit_smoke_*")
    if (path / "checkpoints/best.pt").is_file()
]
if not source_runs:
    raise FileNotFoundError("Nenhum checkpoints/best.pt do overfit foi encontrado")
SOURCE_RUN = max(source_runs, key=lambda path: path.stat().st_mtime)
SOURCE_CHECKPOINT = SOURCE_RUN / "checkpoints/best.pt"
print(f"Checkpoint de origem: {SOURCE_CHECKPOINT}")

run_stream(
    [
        sys.executable,
        "scripts/run_overfit_smoke.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--data-root",
        str(DATA_ROOT),
        "--config",
        "configs/overfit_smoke.yaml",
        "--stabilize-from",
        str(SOURCE_CHECKPOINT),
        "--stabilization-epochs",
        "30",
        "--stabilization-learning-rate",
        "0.0001",
    ],
    allowed_returncodes=(0, 2),
)

## Mostrar o resultado da estabilização

In [ ]:
stabilization_runs = list((PROJECT_ROOT / "results/smoke/overfit").glob("overfit_stabilization_*"))
if not stabilization_runs:
    raise FileNotFoundError("Nenhuma execução de estabilização encontrada")
STABILIZATION_RUN = max(stabilization_runs, key=lambda path: path.stat().st_mtime)
stabilization_summary = json.loads((STABILIZATION_RUN / "final_summary.json").read_text())
display(stabilization_summary)
display(DisplayImage(filename=str(STABILIZATION_RUN / "overfit_panel.png")))
print(f"Resultados: {STABILIZATION_RUN}")

## Retomada após interrupção

Se o Colab for interrompido, execute novamente as células iniciais e use:

```python
RESUME = LATEST_RUN / 'checkpoints/last.pt'
run_stream([sys.executable, 'scripts/run_overfit_smoke.py', '--project-root', str(PROJECT_ROOT), '--data-root', str(DATA_ROOT), '--config', 'configs/overfit_smoke.yaml', '--resume', str(RESUME)])
```